# Decrypting PME-Encrypted Parquet with Athena Spark

This notebook demonstrates reading **Parquet Modular Encryption (PME)** files
written by PyArrow, using Apache Spark in Amazon Athena.

**Encryption details:**
- Algorithm: AES-GCM-V1 (column-level)
- Key management: AWS KMS envelope encryption
- Footer: plaintext (schema discoverable without keys)
- PCI columns (`ssn`): encrypted with `pwe-hackathon-pci-key`
- PII columns (`first_name`, `last_name`, `email`): encrypted with `pwe-hackathon-pii-key`
- Plaintext columns (`xid`, `balance`): not encrypted

**Two decryption approaches shown:**
1. Native Spark reader — reads plaintext columns directly (no keys needed)
2. PyArrow PME — full column-level decryption with AWS KMS

## Prerequisites

The Athena Spark **execution role** (`pwe-hackathon-athena-spark-execution`) needs
KMS Decrypt permission on the PME keys. Add this policy to the role:

```json
{
  "Effect": "Allow",
  "Action": ["kms:Decrypt", "kms:DescribeKey"],
  "Resource": [
    "<footer-key-arn>",
    "<pci-key-arn>",
    "<pii-key-arn>"
  ]
}
```

Without this, Part 1 (plaintext columns) still works, but Part 2+ will fail
with `AccessDeniedException` from KMS.

**Workgroups available:**
- `pwe-hackathon-pme-fraud` — full access (all keys)
- `pwe-hackathon-pme-marketing` — footer + PII keys only
- `pwe-hackathon-pme-junior` — footer key only

In [ ]:
%%configure -f
{
    "conf": {
        "spark.sql.execution.arrow.pyspark.enabled": "true"
    }
}

In [ ]:
# Constants
S3_BUCKET = "pwe-hackathon-pme-data-651767347247"
S3_PREFIX = "pme-data"
REGION    = "us-east-2"

---
## Part 0 — Discovery: Does Athena Spark Have a Built-in AWS KMS Client?

Spark's parquet-mr supports PME but ships **no AWS KMS client** in open-source.
Since Athena is an AWS managed service, AWS *might* have added one to the
classpath. Let's probe before falling back to the PyArrow workaround.

In [ ]:
# Probe 1: Check if AWS added a KMS client class to Athena Spark's classpath
sc = spark.sparkContext

candidates = [
    # AWS-style naming conventions
    "com.amazonaws.athena.parquet.crypto.AwsKmsClient",
    "com.amazonaws.emr.parquet.crypto.AwsKmsClient",
    "software.amazon.parquet.crypto.AwsKmsClient",
    "software.amazon.athena.parquet.crypto.AwsKmsClient",
    # Might live inside the parquet package itself
    "org.apache.parquet.crypto.keytools.AwsKmsClient",
    "org.apache.parquet.crypto.keytools.AWSKmsClient",
]

found_class = None
for cls in candidates:
    try:
        sc._jvm.java.lang.Class.forName(cls)
        print(f"FOUND: {cls}")
        found_class = cls
    except:
        print(f"  not found: {cls}")

if found_class:
    print(f"\n>>> Built-in KMS client detected! Class: {found_class}")
    print(">>> You can use native Spark PME — skip to Part 2b below.")
else:
    print("\n>>> No built-in AWS KMS client found.")
    print(">>> Falling back to PyArrow-in-PySpark approach (Part 2).")

In [ ]:
# Probe 2: List ALL classes in parquet crypto keytools package
# This catches any KMS client we didn't guess the name for.
try:
    import subprocess, re

    # Find the parquet JAR on the classpath
    cp = sc._jvm.java.lang.System.getProperty("java.class.path")
    parquet_jars = [j for j in cp.split(":") if "parquet" in j.lower()]
    print("Parquet JARs on classpath:")
    for j in parquet_jars:
        print(f"  {j}")

    # List classes in the keytools package from each JAR
    for jar in parquet_jars:
        try:
            result = subprocess.run(
                ["jar", "tf", jar],
                capture_output=True, text=True, timeout=10,
            )
            keytools_classes = [
                line.replace("/", ".").replace(".class", "")
                for line in result.stdout.splitlines()
                if "crypto" in line and line.endswith(".class") and "$" not in line
            ]
            if keytools_classes:
                print(f"\nCrypto classes in {jar.split('/')[-1]}:")
                for c in sorted(keytools_classes):
                    print(f"  {c}")
        except Exception:
            pass
except Exception as e:
    print(f"Classpath scan failed: {e}")
    print("(This is expected if 'jar' CLI is not available in Athena's runtime)")

In [ ]:
# Probe 3: If a KMS client WAS found, try native Spark PME decryption
# (Skip this cell if Probe 1 found nothing — it will error.)

if found_class:
    spark.conf.set(
        "spark.hadoop.parquet.crypto.factory.class",
        "org.apache.parquet.crypto.keytools.PropertiesDrivenCryptoFactory",
    )
    spark.conf.set(
        "spark.hadoop.parquet.encryption.kms.client.class",
        found_class,
    )

    # Try reading encrypted columns directly with Spark — no PyArrow needed
    test_uri = f"s3://{S3_BUCKET}/{S3_PREFIX}/"
    df_native = spark.read.parquet(test_uri)
    print("=== Native Spark PME decryption succeeded! ===")
    df_native.show(10, truncate=False)
else:
    print("No built-in KMS client — skipping native Spark PME test.")
    print("Continue to Part 1 for plaintext columns, Part 2 for full decryption.")

---
## Part 1 — Schema Discovery & Plaintext Columns (No Keys Needed)

Because `plaintext_footer=True`, the Parquet footer is readable without any
KMS keys. Spark can discover the schema and read non-encrypted columns.

In [ ]:
# List encrypted Parquet files in S3
import boto3

s3 = boto3.client("s3", region_name=REGION)
response = s3.list_objects_v2(Bucket=S3_BUCKET, Prefix=S3_PREFIX + "/")

parquet_files = [
    obj["Key"] for obj in response.get("Contents", [])
    if obj["Key"].endswith(".parquet")
]

for f in parquet_files:
    print(f"s3://{S3_BUCKET}/{f}")

# Use the first file found (or set explicitly)
PARQUET_KEY = parquet_files[0] if parquet_files else f"{S3_PREFIX}/customer_data_encrypted.parquet"
S3_URI = f"s3://{S3_BUCKET}/{PARQUET_KEY}"
print(f"\nUsing: {S3_URI}")

In [ ]:
# Read only the plaintext (non-encrypted) columns with native Spark
# This works without any KMS keys because these columns are not encrypted.
df_plain = spark.read.parquet(S3_URI).select("xid", "balance")
df_plain.show(10)

In [ ]:
# Print the schema — readable from the plaintext footer
df_schema = spark.read.parquet(S3_URI).select("xid", "balance")
df_schema.printSchema()

---
## Part 2 — Full PME Decryption with PyArrow

To read encrypted columns, we use PyArrow's Parquet Modular Encryption
support with a KMS client that calls AWS KMS to unwrap the data encryption
keys (DEKs) stored in the Parquet footer.

**Flow:**
1. Download encrypted file from S3 to `/tmp/`
2. PyArrow reads wrapped DEK from Parquet footer
3. KMS client calls `kms:Decrypt` to unwrap each DEK
4. PyArrow decrypts column data locally with AES-GCM
5. Convert decrypted PyArrow Table → Spark DataFrame

In [ ]:
import base64
import json
import tempfile
from datetime import timedelta

import pyarrow as pa
import pyarrow.parquet as pq
import pyarrow.parquet.encryption as pe


# ---------------------------------------------------------------------------
# KMS client — bridges PyArrow PME to AWS KMS
# Adapted from pme/src/kms_client.py
# ---------------------------------------------------------------------------

class AwsKmsClient(pe.KmsClient):
    """PyArrow KmsClient backed by AWS KMS.

    wrap_key   → calls KMS Encrypt (used during write)
    unwrap_key → calls KMS Decrypt (used during read)
    """

    def __init__(self, kms_connection_config):
        super().__init__()
        if isinstance(kms_connection_config, pe.KmsConnectionConfig):
            conf = dict(kms_connection_config.custom_kms_conf or {})
        elif isinstance(kms_connection_config, str):
            conf = json.loads(kms_connection_config) if kms_connection_config else {}
        else:
            conf = {}
        region = conf.get("region", REGION)
        self._kms = boto3.client("kms", region_name=region)

    def wrap_key(self, key_bytes: bytes, master_key_identifier: str) -> str:
        """Encrypt a DEK with the specified KMS CMK."""
        # Resolve PME alias to AWS KMS alias format
        key_id = f"alias/{master_key_identifier}"
        resp = self._kms.encrypt(KeyId=key_id, Plaintext=key_bytes)
        return base64.b64encode(resp["CiphertextBlob"]).decode("utf-8")

    def unwrap_key(self, wrapped_key: str, master_key_identifier: str) -> bytes:
        """Decrypt a wrapped DEK using KMS.

        The ciphertext blob already contains the key identifier,
        so KMS can decrypt without us specifying which key.
        """
        ciphertext = base64.b64decode(wrapped_key)
        resp = self._kms.decrypt(CiphertextBlob=ciphertext)
        return resp["Plaintext"]


def kms_client_factory(kms_connection_config):
    """Factory callable for CryptoFactory."""
    return AwsKmsClient(kms_connection_config)


print("KMS client defined.")

In [ ]:
# Download encrypted Parquet from S3 to /tmp/
local_path = f"/tmp/{PARQUET_KEY.split('/')[-1]}"
s3.download_file(S3_BUCKET, PARQUET_KEY, local_path)
print(f"Downloaded to {local_path}")

# Verify magic bytes (PAR1 = plaintext footer, PARE = encrypted footer)
with open(local_path, "rb") as f:
    magic = f.read(4)
print(f"Magic bytes: {magic} ({'plaintext footer' if magic == b'PAR1' else 'encrypted footer'})")

In [ ]:
# Build decryption properties and read all columns
crypto_factory = pe.CryptoFactory(kms_client_factory)

kms_conn_config = pe.KmsConnectionConfig(
    custom_kms_conf={"region": REGION},
)

dec_config = pe.DecryptionConfiguration(
    cache_lifetime=timedelta(seconds=600),
)

dec_props = crypto_factory.file_decryption_properties(kms_conn_config, dec_config)

# Read and decrypt ALL columns
arrow_table = pq.read_table(local_path, decryption_properties=dec_props)

print(f"Decrypted {arrow_table.num_rows} rows, {arrow_table.num_columns} columns")
print(f"Columns: {arrow_table.column_names}")
print()
print(arrow_table.to_pandas().head(10))

In [ ]:
# Convert decrypted PyArrow Table → Spark DataFrame
df_all = spark.createDataFrame(arrow_table.to_pandas())
df_all.show(10, truncate=False)

---
## Part 3 — Selective Column Decryption

PME encrypts each column group with a separate KMS key.
You can decrypt only the columns you have key access to.

| Role | Footer Key | PCI Key | PII Key | Visible Columns |
|------|:---:|:---:|:---:|---|
| **Fraud Analyst** | Y | Y | Y | All |
| **Marketing Analyst** | Y | N | Y | PII + plaintext |
| **Junior Analyst** | Y | N | N | Plaintext only |

In [ ]:
# --- Marketing Analyst view: read only PII + plaintext columns ---
# Skip the PCI column (ssn) entirely — never decrypted, never touched.

arrow_pii = pq.read_table(
    local_path,
    columns=["first_name", "last_name", "email", "xid", "balance"],
    decryption_properties=dec_props,
)

df_marketing = spark.createDataFrame(arrow_pii.to_pandas())
print("=== Marketing Analyst View (PII + plaintext, no PCI) ===")
df_marketing.show(10, truncate=False)

In [ ]:
# --- Junior Analyst view: plaintext columns only ---
# No KMS keys needed at all for this query.

arrow_plain = pq.read_table(
    local_path,
    columns=["xid", "balance"],
    decryption_properties=dec_props,
)

df_junior = spark.createDataFrame(arrow_plain.to_pandas())
print("=== Junior Analyst View (plaintext only) ===")
df_junior.show(10, truncate=False)

---
## Part 4 — Spark SQL on Decrypted Data

Once decrypted into a Spark DataFrame, use standard Spark SQL.

In [ ]:
# Register as a temporary view for SQL queries
df_all.createOrReplaceTempView("customer_data")

# Example queries
print("=== Row count ===")
spark.sql("SELECT COUNT(*) AS total_rows FROM customer_data").show()

print("=== Balance statistics ===")
spark.sql("""
    SELECT
        COUNT(*)          AS total_customers,
        ROUND(AVG(balance), 2)  AS avg_balance,
        ROUND(MIN(balance), 2)  AS min_balance,
        ROUND(MAX(balance), 2)  AS max_balance
    FROM customer_data
""").show()

print("=== Sample decrypted PII + PCI (fraud analyst view) ===")
spark.sql("""
    SELECT first_name, last_name, email, ssn, xid, balance
    FROM customer_data
    LIMIT 5
""").show(truncate=False)

---
## Part 5 — Cleanup

In [ ]:
import os
os.remove(local_path)
print(f"Removed {local_path}")